In [0]:
catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

### Transformações Aplicadas

- Remoção de registros com `id_cliente` nulo.
- Padronização da coluna `nome`, removendo espaços extras e formatando em estilo título.
- Padronização da coluna `email`, convertendo os valores para letras minúsculas e removendo espaços desnecessários.
- Padronização da coluna `regiao`, garantindo nomenclatura consistente.
- Conversão da coluna `data_cadastro` para o tipo `Date`, tratando os formatos encontrados na origem.
- Criação da coluna `ano_cadastro` a partir da data de cadastro.
- Validação da coluna `idade`, mantendo apenas valores entre 0 e 120 anos.
- Validação da coluna `dia_vencimento_fatura`, mantendo apenas valores entre 1 e 31.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df_bronze_clientes = spark.table(
    f"{bronze_schema}.tb_clientes"
)

window_clientes = (
    Window
    .partitionBy("id_cliente")
    .orderBy(
        F.col("ingestion_datetime").desc()
    )
)

tb_clientes = (
    df_bronze_clientes

    .filter(
        F.col("id_cliente").isNotNull()
    )

    .withColumn(
    "row_num",
    F.row_number().over(window_clientes)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")

    .withColumn(
        "nome",
        F.initcap(
            F.trim(F.col("nome"))
        )
    )

    .withColumn(
        "email",
        F.lower(
            F.trim(F.col("email"))
        )
    )

    .withColumn(
        "regiao",
        F.initcap(
            F.trim(F.col("regiao"))
        )
    )

    .withColumn(
        "data_cadastro",
        F.when(
            F.col("data_cadastro").rlike("^\\d{4}-\\d{2}-\\d{2}$"),
            F.to_date(
                F.col("data_cadastro"),
                "yyyy-MM-dd"
            )
        )
        .when(
            F.col("data_cadastro").rlike("^\\d{2}/\\d{2}/\\d{4}$"),
            F.to_date(
                F.col("data_cadastro"),
                "dd/MM/yyyy"
            )
        )
    )

    .withColumn(
        "ano_cadastro",
        F.year("data_cadastro")
    )

    .withColumn(
        "idade",
        F.when(
            (F.col("idade") >= 0) &
            (F.col("idade") <= 120),
            F.col("idade")
        )
    )

    .withColumn(
        "dia_vencimento_fatura",
        F.when(
            (F.col("dia_vencimento_fatura") >= 1) &
            (F.col("dia_vencimento_fatura") <= 31),
            F.col("dia_vencimento_fatura")
        )
    )

    .select(
        "id_cliente",
        "nome",
        "email",
        "regiao",
        "idade",
        "data_cadastro",
        "ano_cadastro",
        "dia_vencimento_fatura"
    )
)

tb_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_clientes"
    )

display(
    spark.table(
        f"{silver_schema}.tb_clientes"
    ).limit(10)
)

id_cliente,nome,email,regiao,idade,data_cadastro,ano_cadastro,dia_vencimento_fatura
10002501082,Sandra Sousa,ssousa@hotmail.com,Centro-oeste,40,2020-08-05,2020,28
10004590307,Camila Jesus Brito,camilabrito@gmail.com,Sudeste,38,2022-11-20,2022,10
10004687197,Daniel Borges,dborges@globo.com,Centro-oeste,39,2025-09-22,2025,5
10007507909,Ana Beatriz Duarte Ferreira,ana.ferreira@gmail.com,Centro-oeste,62,2023-05-03,2023,10
10007648401,Henrique Silva Queiroz,henrique.queiroz37@terra.com.br,Sudeste,51,2025-04-30,2025,20
10007789523,Camila Ferreira Viana,viana.camila@uol.com.br,Sul,77,2026-01-15,2026,10
10010156660,Leandro Camargo,lcamargo@terra.com.br,Sul,55,2023-03-26,2023,15
10010942825,Bruno Dantas,dantas.bruno@hotmail.com,Sudeste,52,2017-04-15,2017,20
10012527977,Nath�lia Gon�alves Moura,moura.nathalia@hotmail.com,Sul,58,2026-01-15,2026,15
10013368321,Pedro Siqueira Almeida,pedro_almeida@hotmail.com,Nordeste,57,2017-04-15,2017,10


### Validações de Qualidade

Após a carga, foram realizadas verificações para:

- Quantidade total de registros.
- Registros com `id_cliente` nulo.
- Registros com `nome` nulo.
- Registros com `email` nulo.
- Existência de registros duplicados por `id_cliente`.

In [0]:
display(
    tb_clientes.select(
        F.count("*").alias("total_registros"),
        F.sum(
            F.col("id_cliente").isNull().cast("int")
        ).alias("id_cliente_nulos"),
        F.sum(
            F.col("nome").isNull().cast("int")
        ).alias("nome_nulos"),
        F.sum(
            F.col("email").isNull().cast("int")
        ).alias("email_nulos")
    )
)

total_registros,id_cliente_nulos,nome_nulos,email_nulos
47705,0,0,0
